# ⚾ Baseball Lab 1
### A mission from MLB scouts: do the biggest home run hitters strike out the most?

Home run hitters swing hard. When you swing hard, you miss more. So **do the players who hit the most home runs
also strike out the most?** Every MLB team has analysts who ask questions like this, because the answer
changes which players a team wants.

You're going to answer it yourself, in Python, with real MLB data: **every season from 1871 to today.**

**How it works:**
- Gray boxes are code. Click a box and press **Shift + Enter** to run it.
- Boxes marked ⚾ **Your turn** are where you change something yourself.
- **Guess first!** Before you run a box, say what you think will happen.
- Some boxes near the end are **"watch the pros"** boxes 👀: you just run them. You don't need to understand
  every line yet. Under each one there's a short explanation in plain words.
- A red error? No big deal, that's how every programmer learns. Read the **last line** of it and try again.

It takes about **an hour**. Mission 6 is a bonus, and you can stop after any mission.

---
## Mission 1. Warm-up 🔥
Python remembers things in **variables**. A variable is like a locker with a name tag: you put something inside,
and later you ask for it by its name. The `=` sign means **"put this into the locker"**.

In [ ]:
player = "Aaron Judge"
team = "Yankees"
number = 99

print(player, "plays for the", team, "and wears number", number)

What's going on here:
- `player = "Aaron Judge"` puts the text **Aaron Judge** into a locker called `player`.
- **Quotes** `" "` mean "this is text". Names and words need quotes. Numbers like `99` don't.
- `print( )` is a **command** that shows things on the screen. Whatever you want to show goes inside the round brackets.
- The **commas** separate the pieces. `player` has **no quotes**, so Python opens that locker and shows what's inside.
  `"plays for the"` **has** quotes, so Python shows those exact words.

### ⚾ Your turn
Change the three values to your favorite player and run it. Keep the quotes around the name and the team.

In [ ]:
player = "Shohei Ohtani"   # change to your favorite player
team = "Dodgers"           # his team
number = 17                # his number

print(player, "plays for the", team, "and wears number", number)

The text after `#` is a **comment**: a note for people. Python ignores it.

Python is also a calculator. Baseball has a stat called **total bases**: a single counts 1, a double 2,
a triple 3, and a home run 4. In Python, `*` means **times** (multiply).

In [ ]:
singles = 2
doubles = 1
triples = 0
home_runs = 1

total_bases = singles * 1 + doubles * 2 + triples * 3 + home_runs * 4
print("Total bases:", total_bases)

`total_bases = ...` does the math first and then puts the answer into a locker called `total_bases`:
2×1 + 1×2 + 0×3 + 1×4 = **8**.

### ⚾ Your turn
Think of a game you watched, or your own game. Change the four numbers to one batter's hits that day.
How many total bases did they get? A really great day is 10 or more.

In [ ]:
singles = 2     # change these numbers
doubles = 1
triples = 0
home_runs = 1

total_bases = singles * 1 + doubles * 2 + triples * 3 + home_runs * 4
print("Total bases:", total_bases)

---
## Mission 2. Opening the secret MLB archive 📂
Now we'll load **every batter's every season, from 1871 to today**. The data comes from two places:
- the **Lahman Baseball Database**, a history book of baseball that researchers have kept for more than 30 years;
- **MLB's own live stats** for the newest season, so your favorite player's numbers from this year are in there too.

Gluing those two together takes a lot of code, so we hid it. 👀 Click the ▶ next to the title below and wait
about 20 seconds. (Curious? Double-click the title to see the code. By the end of these labs you'll be able to read all of it.)

In [ ]:
#@title ⚾ Load the MLB data, 1871 to today (click ▶ and wait about 20 seconds)
import datetime
import unicodedata

import pandas as pd
import requests

LAHMAN = "https://raw.githubusercontent.com/daviddalpiaz/pylahman/main/data-raw/"
MLB = "https://statsapi.mlb.com/api/v1/"
STATS = ["G", "AB", "H", "HR", "RBI", "SB", "BB", "SO"]


def plain(text):
    # "Acuña" -> "Acuna": kids type names on an English keyboard
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()


def short(text):
    # "Acuña Jr." -> "acu": the start of a last name, to match players across the two sources
    return plain(text).lower()[:3]


# History: the Lahman Baseball Database (SABR, CC BY-SA 3.0). One row per player, season and team.
batting = pd.read_csv(LAHMAN + "Batting.csv")
people = pd.read_csv(LAHMAN + "People.csv")
teams = pd.read_csv(LAHMAN + "Teams.csv")[["yearID", "teamID", "name"]].rename(columns={"name": "team"})
batting = batting.merge(teams, on=["yearID", "teamID"], how="left")

# A player traded during a season has a row for each team: add them up and keep his last team.
per_season = batting.groupby(["playerID", "yearID"], sort=False)
history = per_season[STATS].sum(min_count=1).join(per_season["team"].last()).reset_index()
people["name"] = people["nameFirst"] + " " + people["nameLast"]
history = history.merge(people[["playerID", "name"]], on="playerID").rename(columns={"yearID": "season"})

# The seasons the history book doesn't have yet come live from MLB (statsapi.mlb.com).
lahman_ids = {
    (int(p.birthYear), int(p.birthMonth), int(p.birthDay), short(p.nameLast)): p.playerID
    for p in people.dropna(subset=["birthYear", "birthMonth", "birthDay"]).itertuples()
}
mlb_names, live = {}, []
try:
    for season in range(int(history["season"].max()) + 1, datetime.date.today().year + 1):
        url = MLB + f"stats?stats=season&group=hitting&season={season}&sportId=1&playerPool=ALL&limit=5000"
        found = requests.get(url, timeout=60).json().get("stats", [])
        hitters = found[0]["splits"] if found else []
        if not hitters:
            continue
        roster = requests.get(MLB + f"sports/1/players?season={season}", timeout=60).json()["people"]
        born = {p["id"]: p.get("birthDate", "") for p in roster}
        for h in hitters:
            p, s = h["player"], h["stat"]
            y, m, d = (born.get(p["id"]) or "0-0-0").split("-")
            player_id = lahman_ids.get((int(y), int(m), int(d), short(p["lastName"])), f"mlb{p['id']}")
            mlb_names[player_id] = p["fullName"]
            live.append({
                "playerID": player_id, "season": season, "team": h.get("team", {}).get("name"), "name": p["fullName"],
                "G": s["gamesPlayed"], "AB": s["atBats"], "H": s["hits"], "HR": s["homeRuns"], "RBI": s["rbi"],
                "SB": s["stolenBases"], "BB": s["baseOnBalls"], "SO": s["strikeOuts"],
            })
except (requests.RequestException, ValueError, KeyError) as problem:
    live = []
    print("MLB's live stats didn't answer, so the newest season is missing today. Everything else works.", problem)

stats = pd.concat([history, pd.DataFrame(live)], ignore_index=True)
# Players active today get the name MLB uses now on their whole career ("Vladimir Guerrero Jr."), without accents.
stats["name"] = stats["playerID"].map(mlb_names).fillna(stats["name"]).map(plain)
# Whole numbers (114, not 114.0); a stat nobody counted back then stays empty: <NA>.
stats[STATS] = stats[STATS].astype("Int64")
stats = stats.sort_values(["playerID", "season"], ignore_index=True)
stats = stats[["name", "season", "team"] + STATS + ["playerID"]]
print("Ready!", len(stats), "rows, seasons", stats["season"].min(), "to", stats["season"].max())

The box made one big table called `stats`. It uses **pandas**, a helper that lets Python work with tables,
like a spreadsheet. Let's peek at it:
- `len(stats)` **counts** the rows. One row = one player in one season.
- `stats.head()` shows the **first 5 rows**.

In [ ]:
print("Rows in the archive:", len(stats))
stats.head()

What the columns mean:

| column | what it is |
|---|---|
| `name` | player's name |
| `season` | the year |
| `team` | his team that season |
| `G` | games played |
| `AB` | at bats: times he came up to hit (walks don't count) |
| `H` | hits |
| `HR` | home runs |
| `RBI` | runs batted in |
| `SB` | stolen bases |
| `BB` | walks |
| `SO` | strikeouts |
| `playerID` | a secret code for each player, so two players with the same name don't get mixed up |

`<NA>` means **empty**: there's no number there. In the very old days, some stats weren't counted.

### ⚾ Your turn: find your player
Read the code like a sentence: **"from `stats`, keep the rows where `name` is the same as the name I picked."**
- `stats["name"]` with square brackets means "the `name` column".
- `==` (two equal signs) **asks a question**: "are these the same?" One `=` puts into a locker; two `==` compare.
- `stats` is written twice: the inside one is where we **ask** the question about every row,
  and the outside one **keeps** the rows where the answer is yes. Lab 2 takes this line apart step by step.

Change the name to your player, spelled **exactly** (capital letters matter). Run it.
(The double square brackets at the end pick which columns to show. Mission 3 says what they are.)

In [ ]:
name = "Aaron Judge"   # change this

stats[stats["name"] == name][["name", "season", "team", "G", "H", "HR", "RBI", "SB", "SO"]]

Sometimes two players share a name. Try `"Ken Griffey"`: that's a dad **and** his son, who both played!
Today's players have their names written the way MLB writes them, like `"Vladimir Guerrero Jr."`
(and `"Vladimir Guerrero"` is his dad).

All names are written **without accent marks**, so you can type them on any keyboard:
`"Julio Rodriguez"`, `"Ronald Acuna Jr."`, `"Teoscar Hernandez"`.

---
## Mission 3. The most home runs ever ⚾💥
This box puts every season in order by home runs:
- `stat = "HR"` puts a column name into a locker. After that, `stat` has **no quotes**, so Python opens the locker
  and uses `HR`. That's why you'll only need to change one line to look at a different stat.
- `.sort_values(stat, ascending=False)` puts the rows in order. `ascending=False` means **biggest first**.
  Here `=` is a **setting**, like a switch on the command.
- `[["name", "season", "team", stat]]`, **double** square brackets, picks which columns to show. They are really two things: the inside `[ ]` is a **list**
  (a few things in a row, with commas between them), and the outside `[ ]` means "pick from the table". Lab 2 shows you more.
- `.head(10)` shows the **top 10**.

In [ ]:
stat = "HR"

top = stats.sort_values(stat, ascending=False)
top[["name", "season", "team", stat]].head(10)

### ⚾ Your turn: other records
Guess first: who has the most **stolen bases** in one season? Then change **only the first line**:
`"HR"` → `"SB"`. Keep the quotes. Then try `"H"` (hits) and `"SO"` (strikeouts).

(Some stolen base records are from the 1880s, when the rules were different. Baseball is old!)

In [ ]:
stat = "HR"   # try "SB", "H", "SO", "RBI"

top = stats.sort_values(stat, ascending=False)
top[["name", "season", "team", stat]].head(10)

Now just one season. This box keeps only one year first, and then sorts it.
Its last line is a chain, read from left to right: sort, pick the columns, show the top 10.
### ⚾ Your turn
It starts with **2026**. Then change `2026` to the year you were born, or to **1998**:
that summer two players raced each other for the record. Numbers don't need quotes.

In [ ]:
year = 2026   # change the year

one_year = stats[stats["season"] == year]
one_year.sort_values("HR", ascending=False)[["name", "team", "HR"]].head(10)

---
## Mission 4. Fastball timer ⏱️
It's **60 feet 6 inches** (60.5 feet) from the pitcher's mound to home plate. If you know how fast a pitch is,
you can work out how long the batter has to decide whether to swing.
We'll build our own **machine** (programmers call it a **function**): you put a speed in, and a time comes out.

In [ ]:
def time_to_plate(mph):
    feet_per_second = mph * 5280 / 3600
    return round(60.5 / feet_per_second, 2)

print("A 100 mph fastball reaches home plate in", time_to_plate(100), "seconds")

How the machine works, line by line:

| line | what it means |
|---|---|
| `def time_to_plate(mph):` | "build a machine called `time_to_plate`. You feed it a number, and inside it's called `mph`." |
| `feet_per_second = mph * 5280 / 3600` | a mile has 5,280 feet and an hour has 3,600 seconds, so this turns miles per hour into feet in one second. `/` means **divide**. |
| `return round(60.5 / feet_per_second, 2)` | 60.5 feet ÷ the feet in one second = the seconds the ball needs. `round(..., 2)` keeps 2 numbers after the dot. `return` hands the answer back. |

The lines inside the machine are pushed to the right (**indented**): that's how Python knows they belong to it.
Then `time_to_plate(100)` **uses** the machine: 100 goes in as `mph`, and the answer comes out.

**0.41 seconds.** A blink takes about a third of a second. So a batter has about **one blink** to see the pitch,
decide, and swing! (Pitchers actually let go of the ball a few feet in front of the mound, so it's even less.)

### ⚾ Your turn
Guess each time before you run it. Change **only the numbers inside** `time_to_plate( )`:
1. `105.8`: Aroldis Chapman's pitch in 2010, the fastest ever measured.
2. `80`: a big-league curveball.
3. `45`: about how fast a 10-year-old pitches. How fast can **you** throw?
   (A Little League mound is only 46 feet from home plate, not 60.5, so the real time for a kid's pitch is shorter.
   That's exactly why kids pitch from closer!)

In [ ]:
print(time_to_plate(105.8), "seconds")   # change the number

---
## Mission 5. The big question 🔬
Back to the scouts' question: **do the biggest home run hitters strike out the most?**

### Step 1 of 3: whole careers (👀 watch the pros)
Right now one row = one season. We need one row = a **whole career**.
This box adds up all of a player's seasons, then keeps only players with at least **4,000 at bats**
(about 7 or 8 full seasons), so we only look at players who played a lot. Just run it:

In [ ]:
counted = stats.dropna(subset=["SO"])
careers = counted.groupby(["playerID", "name"], as_index=False)[["AB", "HR", "SO"]].sum()
careers = careers[careers["AB"] >= 4000]

print("Players with at least 4,000 at bats:", len(careers))
careers.sort_values("HR", ascending=False)[["name", "HR", "SO"]].head(10)

What it did:
- `.dropna(subset=["SO"])` throws away the seasons where `SO` is empty. Long ago, and in many Negro Leagues games,
  nobody wrote down strikeouts. If we kept those seasons, a great old hitter would look like he never struck out!
  `subset=` is a **setting**, like a switch on the command: it says which column to check.
- `groupby` collects all the rows that have the same player, and `.sum()` adds up their numbers.
  `["playerID", "name"]` is a **list**, so Python can use both: the secret code keeps Ken Griffey the dad and Ken Griffey the son apart.
- The last line keeps only careers where `AB` is **at least** 4,000 (`>=` means "at least").

### Step 2 of 3: guess first! 🤔
We'll draw one dot per player:
- **across** (left to right): career home runs. **Right = more home runs.**
- **up** (bottom to top): career strikeouts. **Up = more strikeouts.**

If home run hitters strike out the most, the dots should go **up as they go right**, like a hill.
What do you think: hill or no hill? And will the top home run hitters be the very highest dots?

### Step 3 of 3: the chart (👀 watch the pros)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 7))
plt.scatter(careers["HR"], careers["SO"], alpha=0.5)

for _, row in careers.sort_values("HR", ascending=False).head(8).iterrows():
    plt.annotate(row["name"], (row["HR"], row["SO"]), fontsize=8)

plt.xlabel("Career home runs   (right = more home runs)")
plt.ylabel("Career strikeouts   (up = more strikeouts)")
plt.title("Do home run hitters strike out more?")
plt.show()

What the box did: `matplotlib` is a drawing helper. `scatter` drew one dot per player,
the `for` line wrote the names of the top 8 home run hitters, and the last lines added the labels and showed the chart.

Three strange-looking bits, if you're curious:
- `figsize=(14, 7)` is a setting for how big the chart is: how wide, then how tall.
- `alpha=0.5` makes the dots half see-through, so you can spot dots sitting on top of each other.
- `for _, row in ...` goes through the top players one at a time. You'll learn this in Lab 5.
  `_` is a way of saying "I don't need this part".

### ⚾ Your turn: be the scout
Look at the chart and answer out loud (or type it here: double-click this box):
1. Do the dots go up as they go right?
2. Look at the names on the far right. Are they the **highest** dots, or is someone higher?
3. Was your guess right?

Run the next box to see the home run kings who **didn't** strike out much: 500 or more home runs,
but fewer than 1,450 strikeouts. (`<` means "less than", and `&` means both must be true.)

It's the longest line so far, but it's only three steps:
1. `(careers["HR"] >= 500)` and `(careers["SO"] < 1450)` each ask every row a question.
   The **round brackets** keep each question in one piece, so Python doesn't mix them up.
2. `&` means a row must say `True` to **both** questions.
3. `careers[ ... ]` keeps those rows, and the `[[ ]]` at the end picks the columns to show.

Lab 2 takes a shorter line like this apart, step by step.

In [ ]:
careers[(careers["HR"] >= 500) & (careers["SO"] < 1450)][["name", "HR", "SO"]]

And here are players who struck out a lot but didn't hit many home runs: 1,800 or more strikeouts,
but fewer than 300 home runs. Do you know one of them? Hint: he was a famous Yankees shortstop.

In [ ]:
careers[(careers["SO"] >= 1800) & (careers["HR"] < 300)][["name", "HR", "SO"]]

**So what's the answer?** Mostly **yes**: the dots go up like a hill, so players who hit more home runs
usually strike out more. Swinging hard means missing more.

But look at the far right. Barry Bonds, Hank Aaron and Babe Ruth hit more home runs than anyone,
and they struck out **less** than Jim Thome or Alex Rodriguez. Hitting the ball far **without** missing much
is super rare, and that's what made them legends. (Also, players in the old days struck out less in general:
Ted Williams has only 709!)

---
## Bonus mission 6. What about speed? 🧪
Only if you still have energy! Do the fastest base stealers also hit home runs?

👀 This box is Mission 5's boxes squeezed into one. The first two lines say **which stat goes across**
and **which stat goes up**; the rest works the same way. It writes the names of the top players in both stats.
(In the label lines, `+` glues pieces of text together.) Just run it:

In [ ]:
across = "SB"
up = "HR"

both = stats.dropna(subset=[across, up]).groupby(["playerID", "name"], as_index=False)[["AB", across, up]].sum()
both = both[both["AB"] >= 4000]

plt.figure(figsize=(14, 7))
plt.scatter(both[across], both[up], alpha=0.5)
for _, row in both.sort_values(up, ascending=False).head(5).iterrows():
    plt.annotate(row["name"], (row[across], row[up]), fontsize=8)
for _, row in both.sort_values(across, ascending=False).head(5).iterrows():
    plt.annotate(row["name"], (row[across], row[up]), fontsize=8)
plt.xlabel(across + "   (right = more)")
plt.ylabel(up + "   (up = more)")
plt.title("Career " + across + " and " + up)
plt.show()

No hill this time: the big base stealers are low, and the big home run hitters are on the left.
Most players are good at **one** of these. Now look at two names: Barry Bonds (at the top) and
Rickey Henderson (far right). How many of the **other** stat did each of them get?

### ⚾ Your turn
Now: do home run hitters get more **walks**? (Pitchers are scared to throw them strikes.)
In the box below, change **only the first two lines**:
- `"SB"` → `"HR"`
- `"HR"` → `"BB"`

Keep the quotes! Then press Shift + Enter. Hill or no hill?

In [ ]:
across = "SB"   # change to "HR"
up = "HR"       # change to "BB"

both = stats.dropna(subset=[across, up]).groupby(["playerID", "name"], as_index=False)[["AB", across, up]].sum()
both = both[both["AB"] >= 4000]

plt.figure(figsize=(14, 7))
plt.scatter(both[across], both[up], alpha=0.5)
for _, row in both.sort_values(up, ascending=False).head(5).iterrows():
    plt.annotate(row["name"], (row[across], row[up]), fontsize=8)
for _, row in both.sort_values(across, ascending=False).head(5).iterrows():
    plt.annotate(row["name"], (row[across], row[up]), fontsize=8)
plt.xlabel(across + "   (right = more)")
plt.ylabel(up + "   (up = more)")
plt.title("Career " + across + " and " + up)
plt.show()

---
### 🏆 You finished Baseball Lab 1!
You just did what MLB team analysts do: load data, ask a question, draw a chart, and find the answer.

Today you used:
- **variables** (lockers with names) and `print( )`, with **quotes** for text
- `*` and `/` to do math
- a `pandas` table: `len( )` to count, `.head()` to peek, `.dropna()` to skip empty numbers
- `==` to find a player, and `sort_values` to put seasons in order
- your own **machine** (`def`) that turns a pitch speed into seconds

Show your chart to someone and explain what you found. Next, in **Lab 2**, we slow down and learn exactly
what every symbol does, so you can write more of the code yourself.